# FlowLOT benchmark pipeline
This notebook demonstrates Stage 2 inspection, marker preprocessing, patient/barycenter references, LOT computation, multi-tube fusion, classification/regression, and publication exports. Change the configuration cell for any conforming cohort.

In [ ]:
from pathlib import Path
import numpy as np
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import cross_val_predict, StratifiedKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from flowlot.io import Stage2Loader, Stage2Organizer
from flowlot.transport import compute_stage2_embeddings
from flowlot.models.fusion import EarlyTubeFusion
from flowlot.evaluation.metrics import classification_metrics, regression_metrics
from flowlot.evaluation.reporting import plot_classification_suite, plot_regression_suite, plot_transport_geometry
from flowlot.evaluation.table_exporter import export_csv, export_latex, summarize_runs

## 1. Configuration
`REFERENCE` may be `patient0`, `pooled`, `gaussian`, `uniform`, or `barycenter`; `SOLVER` may be `sinkhorn`, `emd`, `linprog`, or equal-size `hungarian`.

In [ ]:
STAGE2 = Path('stage2_analytics.h5')
DATASET = 'BLAST110'
CELL_COUNT = '1000'
PREPROCESS_ID = 'common12'
REFERENCE = 'patient0'  # compare with 'barycenter'
SOLVER = 'sinkhorn'
selected_markers = ['FSC-A', 'FSC-H', 'SSC-A', 'SSC-H', 'FITC-A', 'PE-A', 'PerCP-A', 'PC7-A', 'APC-A', 'APC-H7-A', 'Horizon V450-A', 'Horizon V500-A']
RESULTS = Path('notebook_results')
RESULTS.mkdir(exist_ok=True)

## 2. Inspect and preprocess Stage 2

In [ ]:
with Stage2Loader(STAGE2) as data:
    tubes = data.tubes(DATASET, CELL_COUNT)
    display({tube: data.metadata(DATASET, CELL_COUNT, tube) for tube in tubes})
# Run once if the preprocess group does not exist:
# Stage2Organizer('unused', STAGE2).add_preprocess(DATASET, CELL_COUNT, PREPROCESS_ID, selected_markers, arcsinh_cofactor=5)

## 3. Compute LOT embeddings

In [ ]:
shapes = compute_stage2_embeddings(
    STAGE2, DATASET, CELL_COUNT, PREPROCESS_ID,
    reference_type=REFERENCE, solver=SOLVER, reference_size=256,
    solver_kwargs={'reg': 0.01}, random_state=42,
)
shapes

## 4. Load tubes and perform early fusion
Missing tubes are imputed from training means and accompanied by availability indicators. For rigorous studies fit the fusion object independently inside every CV fold, as `flowlot-eval` does.

In [ ]:
embedding_id = f'{REFERENCE}_{SOLVER}'
features, labels = {}, {}
with Stage2Loader(STAGE2) as data:
    for tube in data.tubes(DATASET, CELL_COUNT):
        ids, z = data.embeddings(DATASET, CELL_COUNT, tube, PREPROCESS_ID, embedding_id)
        metadata = data.metadata(DATASET, CELL_COUNT, tube)
        tube_labels = dict(zip(metadata['patient_ids'], metadata['labels']))
        features[tube] = dict(zip(ids, z))
        labels.update({patient: tube_labels[patient] for patient in ids})
patient_ids = sorted(labels)
X = EarlyTubeFusion('mean', add_indicators=True).fit_transform(features, patient_ids)
y = np.asarray([labels[patient] for patient in patient_ids])
X.shape, y.shape

## 5A. Classification benchmark

In [ ]:
classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
cv = StratifiedKFold(5, shuffle=True, random_state=42)
probabilities = cross_val_predict(classifier, X, y.astype(int), cv=cv, method='predict_proba')
classification_result = classification_metrics(y.astype(int), probabilities)
plot_classification_suite(y.astype(int), probabilities, RESULTS / 'classification')
classification_result

## 5B. Regression/quantification benchmark
Set `continuous_y` to blast%, LAIP/WBC%, MRD, or another continuous Stage 2 target. The legacy experiments commonly modeled `log10(percent + 1e-8)`. 

In [ ]:
# continuous_y = np.log10(np.asarray(blast_percent) + 1e-8)
# regressor = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
# predictions = cross_val_predict(regressor, X, continuous_y, cv=KFold(5, shuffle=True, random_state=42))
# regression_result = regression_metrics(continuous_y, predictions)
# plot_regression_suite(continuous_y, predictions, RESULTS / 'quantification', 'log10 blast %')
# regression_result

## 6. Transport geometry and table exports

In [ ]:
with Stage2Loader(STAGE2) as data:
    tube = data.tubes(DATASET, CELL_COUNT)[0]
    patient = data.metadata(DATASET, CELL_COUNT, tube)['patient_ids'][0]
    sample = data.cells(DATASET, CELL_COUNT, tube, patient, PREPROCESS_ID)
import h5py
with h5py.File(STAGE2, 'r') as h5:
    root = h5[f'{DATASET}/{CELL_COUNT}/{tube}/lot_embeddings/{PREPROCESS_ID}/{embedding_id}']
    reference = root['reference_matrix'][...]
    transported = root[f'sorted_cell_matrices/{patient}'][...]
plot_transport_geometry(reference, sample, transported, RESULTS / 'transport_geometry')
rows = [{'model': f'LOT-{REFERENCE}-{SOLVER}', 'fold': 0, **classification_result}]
export_csv(rows, RESULTS / 'metrics.csv')
summary = summarize_runs(rows)
export_latex(summary, RESULTS / 'metrics.tex')